In [1]:
import os  
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# ---------------------------
# 1. 定义加密货币及其类型
# ---------------------------
crypto_types = {
    "Bitcoin": "brown",            # Bitcoin（对应原 BTC）
    "Cardano": "green",            # Cardano（对应 ADA）
    "Cosmos": "green",             # Cosmos（对应 ATOM）
    "Dogecoin": "brown",           # Dogecoin
    "Ethereum Classic": "brown",   # Ethereum Classic（对应 ETC）
    "Ethereum": "brown",           # Ethereum（对应 ETH，按字典规定）
    "Litecoin": "brown",           # Litecoin
    "Monero": "brown",             # Monero
    "Polygon": "green",            # Polygon（对应 MATIC）
    "Solana": "green",             # Solana
    "Stellar": "green",            # Stellar（对应 XLM）
    "Tron": "green",               # Tron（对应 TRX）
    "Ripple": "green"              # Ripple（对应 XRP）
}

# ---------------------------
# 2. 指定数据文件所在目录
# ---------------------------
directory = "D:\桌面\绿色货币\数据总库\加密货币数据\coinmarketcap\weekly"  # 修改为你的实际路径

# ---------------------------
# 3. 遍历目录读取 CSV 并合并数据
# ---------------------------
dfs = []
selected_columns = ['name', 'open', 'high', 'low', 'close', 'volume', 'marketCap', 'timestamp']

for f in os.listdir(directory):
    # 只处理 CSV 文件且文件名开头为 crypto_types 中的币种名称
    ticker = f.split("_")[0]
    if f.endswith(".csv") and ticker in crypto_types:
        file_path = os.path.join(directory, f)
        try:
            df = pd.read_csv(file_path, sep=";", quotechar='"', parse_dates=["timestamp"])
            # 若存在不需要的列，如 "Start"，则删除
            df.drop(columns=["Start"], errors="ignore", inplace=True)
            # 仅保留指定列（注意：如果 CSV 中列名不同，需要进行相应调整或重命名）
            df = df[selected_columns]
            # 添加币种信息
            df["source"] = ticker
            df["Type"] = crypto_types[ticker]
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

# 合并所有读取的 DataFrame
data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
# 格式化 timestamp，保留 YYYY-MM-DD 形式
data['timestamp'] = pd.to_datetime(data['timestamp']).dt.strftime('%Y-%m-%d')

# 查看合并后的数据预览
data.rename(columns={"timestamp": "Date"}, inplace=True)

# ---------------------------
# 2. 处理数据格式
# ---------------------------
df = data.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df[df["Date"] >= "2019-06-30"]  # 筛选时间段，可根据需要调整
df = df.sort_values(["source", "Date"]).reset_index(drop=True)

# ---------------------------
# 3. 计算对数收益率（rtn）及下一期收益率（rtn_next）
# ---------------------------
df["rtn"] = df.groupby("source")["close"].transform(lambda x: np.log(x).diff())
df["rtn_next"] = df.groupby("source")["rtn"].shift(-1)
df = df.dropna()

# ---------------------------
# 4. Construct the three factors using information at time t
# ---------------------------
# 4.1 CMKT (market factor) using current period returns (rtn)
CMKT = df.groupby("Date").apply(lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"]))
CMKT.name = "CMKT"
CMKT

<>:29: SyntaxWarning: invalid escape sequence '\c'
<>:29: SyntaxWarning: invalid escape sequence '\c'
C:\Users\why97\AppData\Local\Temp\ipykernel_11008\2793983966.py:29: SyntaxWarning: invalid escape sequence '\c'
  directory = "D:\桌面\绿色货币\数据总库\加密货币数据\coinmarketcap\weekly"  # 修改为你的实际路径
C:\Users\why97\AppData\Local\Temp\ipykernel_11008\2793983966.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMKT = df.groupby("Date").apply(lambda g: np.sum(g["rtn"] * g["marketCap"]) / np.sum(g["marketCap"]))


Date
2019-07-07    0.050633
2019-07-14   -0.147687
2019-07-21    0.034321
2019-07-28   -0.097442
2019-08-04    0.117301
                ...   
2024-12-01    0.034584
2024-12-08    0.052336
2024-12-15    0.007457
2024-12-22   -0.117766
2024-12-29   -0.009032
Name: CMKT, Length: 287, dtype: float64

In [2]:
import pandas as pd
import numpy as np

# 4.3 动量因子 CMOM（采用三周期动量） 
# 4.2 CSMB (size factor)
df_size = df.groupby("Date", group_keys=False).apply(
    lambda sub: sub.assign(Size_Group=pd.qcut(sub["marketCap"], [0, 0.3, 0.7, 1.0], labels=["Small", "Middle", "Big"]))
).reset_index(drop=True)

CSMB = (df_size[df_size["Size_Group"] == "Small"].groupby("Date")["rtn"].mean() -
        df_size[df_size["Size_Group"] == "Big"].groupby("Date")["rtn"].mean())
CSMB.name = "CSMB"
# 先计算滚动三期动量：momentum = (1+r1)*(1+r2)*(1+r3)-1
df_size["momentum"] = df_size.groupby("source")["rtn"].transform(
    lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1)
df_size = df_size.dropna()

# 只保留 Small 和 Big 两组数据
df_mom = df_size[df_size["Size_Group"].isin(["Small", "Big"])].copy()

# 定义手动排序的分组函数
def assign_momentum_group_sorted(sub):
    # 对当前分组按 momentum 升序排序，并重置索引
    sub = sub.sort_values("momentum").reset_index(drop=True)
    n = len(sub)
    idx_low = int(np.floor(n * 0.3))
    idx_high = int(np.floor(n * 0.7))
    
    sub.loc[sub.index[:idx_low], "Momentum_Group"] = "Low"
    sub.loc[sub.index[idx_low:idx_high], "Momentum_Group"] = "Middle"
    sub.loc[sub.index[idx_high:], "Momentum_Group"] = "High"
    
    return sub

# 按照 Date 与 Size_Group 分组，并对每个分组应用手动排序分组方法
df_mom = df_mom.groupby(["Date", "Size_Group"], group_keys=False).apply(assign_momentum_group_sorted)

# 定义函数计算每个日期的动量因子 CMOM，按照公式计算：
# CMOM = 1/2*(Small High + Big High) - 1/2*(Small Low + Big Low)
def compute_cmom(group):
    # 分别提取各组的平均收益率
    small_high = group[(group["Size_Group"]=="Small") & (group["Momentum_Group"]=="High")]["rtn"].mean()
    big_high   = group[(group["Size_Group"]=="Big")   & (group["Momentum_Group"]=="High")]["rtn"].mean()
    small_low  = group[(group["Size_Group"]=="Small") & (group["Momentum_Group"]=="Low")]["rtn"].mean()
    big_low    = group[(group["Size_Group"]=="Big")   & (group["Momentum_Group"]=="Low")]["rtn"].mean()
    
    # 如果任一组数据缺失，则返回 NaN
    if pd.isnull(small_high) or pd.isnull(big_high) or pd.isnull(small_low) or pd.isnull(big_low):
        return np.nan
    
    cmom = 0.5 * (small_high + big_high) - 0.5 * (small_low + big_low)
    return cmom

# 重置 df_mom 的索引，避免 "Date" 同时出现在索引和列中
df_mom = df_mom.reset_index(drop=True)

# 按日期分组计算 CMOM
CMOM = df_mom.groupby("Date").apply(compute_cmom)
CMOM.name = "CMOM"

# 输出结果
print(CMOM)


C:\Users\why97\AppData\Local\Temp\ipykernel_11008\3584091737.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_size = df.groupby("Date", group_keys=False).apply(
C:\Users\why97\AppData\Local\Temp\ipykernel_11008\3584091737.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_mom = df_mom.groupby(["Date", "Size_Group"], group_keys=False).apply(assign_momentum_group_sorted)
C:\Users\why97\AppData\Local\Temp\ipykernel_11008\3584091737.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the gro

Date
2019-07-21    0.093809
2019-07-28    0.037406
2019-08-04    0.098043
2019-08-11    0.149171
2019-08-18    0.056093
                ...   
2024-12-01    0.187131
2024-12-08   -0.035160
2024-12-15    0.038315
2024-12-22    0.129359
2024-12-29   -0.016461
Name: CMOM, Length: 285, dtype: float64


C:\Users\why97\AppData\Local\Temp\ipykernel_11008\3584091737.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMOM = df_mom.groupby("Date").apply(compute_cmom)


In [3]:
# ---------------------------
# 5. Read 1-month US Treasury yield data FRED and calculate excess returns
# ---------------------------
rf_file_path = r"D:\桌面\绿色货币\数据总库\宏观经济数据\美国国债\us_treasury_daliy_1M.csv"  # Update the path as needed
rf_data = pd.read_csv(rf_file_path)
rf_data["Date"] = pd.to_datetime(rf_data["Date"])
rf_data["1 Mo"] = rf_data["1 Mo"] / 100  # Convert to decimals
rf_data = rf_data.rename(columns={"1 Mo": "Rf"})
rf_data = rf_data.set_index("Date").resample("W").mean().reset_index()
# # Merge risk-free rate with df based on current period End
df = df.merge(rf_data, on="Date", how="left")
df["Rf_next"] = df.groupby("source")["Rf"].shift(-1)
# # rp_next = rtn_next - Rf_next
df["rp_next"] = df["rtn_next"] - df["Rf_next"]
df = df.dropna()
df

,name,open,high,low,close,volume,marketCap,Date,source,Type,rtn,rtn_next,Rf,Rf_next,rp_next
0,2781,10796.930618,12006.075356,9737.884544,11450.846755,1.936904e+10,2.038819e+11,2019-07-07,Bitcoin,brown,0.056930,-0.110195,0.022225,0.021920,-0.132115
1,2781,11446.597155,13129.528934,10234.576414,10256.058276,2.248600e+10,1.827419e+11,2019-07-14,Bitcoin,brown,-0.110195,0.032901,0.021920,0.021360,0.011541
2,2781,10257.838536,11052.766797,9163.134644,10599.105396,1.713058e+10,1.889875e+11,2019-07-21,Bitcoin,brown,0.032901,-0.103929,0.021360,0.021280,-0.125209
3,2781,10596.948075,10651.791202,9252.296863,9552.860435,1.373869e+10,1.704620e+11,2019-07-28,Bitcoin,brown,-0.103929,0.138340,0.021280,0.020840,0.117500
4,2781,9548.178894,11009.206914,9437.336287,10970.184216,1.653089e+10,1.959079e+11,2019-08-04,Bitcoin,brown,0.138340,0.049214,0.020840,0.020560,0.028654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3683,2781,0.199486,0.224408,0.193325,0.208833,7.948574e+09,1.802931e+10,2024-11-24,Tron,green,0.045778,-0.007172,0.046980,0.047500,-0.054672
3684,2781,0.208832,0.212175,0.186791,0.207340,6.611200e+09,1.789464e+10,2024-12-01,Tron,green,-0.007172,0.430276,0.047500,0.046440,0.383836
3685,2781,0.207340,0.440675,0.203890,0.318823,3.291900e+10,2.750537e+10,2024-12-08,Tron,green,0.430276,-0.111109,0.046440,0.044880,-0.155989
3686,2781,0.318832,0.320485,0.245066,0.285296,1.512752e+10,2.460455e+10,2024-12-15,Tron,green,-0.111109,-0.156822,0.044880,0.044360,-0.201182


In [4]:
# ---------------------------
# 6. Prepare factor DataFrame (using current period t factors)
# ---------------------------
test_factors = pd.concat([CMKT, CSMB, CMOM], axis=1).dropna().reset_index()
# 使用当前 t 时点的风险溢价来调整 CMKT（构造超额市场因子）
test_factors["prev_date"] = test_factors["Date"] - pd.Timedelta(days=1)

test_factors = pd.merge_asof(
    test_factors.sort_values("prev_date"),
    rf_data.sort_values("Date"),
    left_on="prev_date",
    right_on="Date",
    direction="backward"
)
test_factors.drop(columns=["prev_date", "Date_y"], inplace=True)
test_factors.rename(columns={"Date_x": "Date"}, inplace=True)
test_factors["CMKT"] = test_factors["CMKT"] - test_factors["Rf"]

# ---------------------------
# 7. Perform regression analysis using next period excess returns (rp_next)
# ---------------------------
# 这里合并时用的是当前时点的因子，与各资产下一期的超额收益对应
df_reg = pd.merge(df, test_factors, on="Date", how="inner")

In [5]:

# 使用下一期超额收益 rp_next 作为被解释变量进行回归
green_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "green"]).fit()
brown_model = smf.ols("rp_next ~ CMKT + CSMB + CMOM", data=df_reg[df_reg["Type"] == "brown"]).fit()

print(green_model.summary())
print(brown_model.summary())

alpha_green = green_model.params["Intercept"]
se_green = green_model.bse["Intercept"]
alpha_brown = brown_model.params["Intercept"]
se_brown = brown_model.bse["Intercept"]

t_stat = (alpha_green - alpha_brown) / np.sqrt(se_green**2 + se_brown**2)
df_eff = min(green_model.df_resid, brown_model.df_resid)
p_val = 2 * (1 - t_dist.cdf(abs(t_stat), df=df_eff))

print("\nAlpha difference test between Green and Brown groups:")
print(f"Green alpha: {alpha_green:.4f} (SE={se_green:.4f})")
print(f"Brown alpha: {alpha_brown:.4f} (SE={se_brown:.4f})")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")


                            OLS Regression Results                            
Dep. Variable:                rp_next   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                     21.47
Date:                Fri, 14 Mar 2025   Prob (F-statistic):           1.12e-13
Time:                        19:15:16   Log-Likelihood:                 925.23
No. Observations:                1934   AIC:                            -1842.
Df Residuals:                    1930   BIC:                            -1820.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0163      0.004     -3.624      0.0